In [1]:
# CELL 1 - Create the FastAPI main.py file
main_py = '''from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List
import joblib
import numpy as np
import pandas as pd

app = FastAPI(title="Churn Prediction API")

model = joblib.load("model.pkl")

class CustomerFeatures(BaseModel):
    recency_days: float
    frequency_180d: float
    monetary_180d: float
    return_rate_180d: float
    avg_discount_pct_180d: float
    avg_rating_180d: float
    category_diversity_180d: float
    ticket_count_90d: float
    negative_ticket_rate_90d: float
    avg_resolution_hours_90d: float
    days_since_signup: float
    sessions_30d: float
    product_views_30d: float
    cart_adds_30d: float
    wishlist_adds_30d: float
    abandoned_carts_30d: float
    email_opens_30d: float
    campaign_clicks_30d: float
    last_visit_days_ago: float

def get_risk_level(prob):
    if prob >= 0.7:
        return "high"
    elif prob >= 0.4:
        return "medium"
    else:
        return "low"

def get_explanation(prob, recency, sessions):
    if prob >= 0.7:
        return f"High churn risk. Customer inactive for {recency} days with only {sessions} sessions."
    elif prob >= 0.4:
        return f"Medium churn risk. Recency of {recency} days and {sessions} sessions in last 30 days."
    else:
        return f"Low churn risk. Customer is active with {sessions} sessions recently."

@app.get("/health")
def health():
    return {"status": "ok", "model": "Random Forest Churn Predictor"}

@app.post("/predict")
def predict(customer: CustomerFeatures):
    try:
        features = pd.DataFrame([customer.dict()])
        prob = model.predict_proba(features)[0][1]
        predicted_class = int(prob >= 0.4)
        return {
            "churn_probability": round(float(prob), 4),
            "predicted_class": predicted_class,
            "risk_level": get_risk_level(prob),
            "risk_explanation": get_explanation(prob, customer.recency_days, customer.sessions_30d)
        }
    except Exception as e:
        raise HTTPException(status_code=400, detail=str(e))

@app.post("/batch_predict")
def batch_predict(customers: List[CustomerFeatures]):
    results = []
    for customer in customers:
        features = pd.DataFrame([customer.dict()])
        prob = model.predict_proba(features)[0][1]
        predicted_class = int(prob >= 0.4)
        results.append({
            "churn_probability": round(float(prob), 4),
            "predicted_class": predicted_class,
            "risk_level": get_risk_level(prob),
            "risk_explanation": get_explanation(prob, customer.recency_days, customer.sessions_30d)
        })
    return {"predictions": results, "total": len(results)}
'''

with open('/Users/pratikshaupadhyay/Desktop/capstone/part4/app/main.py', 'w') as f:
    f.write(main_py)
print("main.py created!")

main.py created!


In [2]:
# CELL 2 - Create test file and other required files

# Test file
test_py = '''import requests
import json

BASE_URL = "http://localhost:8000"

sample_customer = {
    "recency_days": 45,
    "frequency_180d": 3,
    "monetary_180d": 1500.0,
    "return_rate_180d": 0.1,
    "avg_discount_pct_180d": 0.2,
    "avg_rating_180d": 3.5,
    "category_diversity_180d": 2,
    "ticket_count_90d": 1,
    "negative_ticket_rate_90d": 0.0,
    "avg_resolution_hours_90d": 24.0,
    "days_since_signup": 365,
    "sessions_30d": 3,
    "product_views_30d": 10,
    "cart_adds_30d": 2,
    "wishlist_adds_30d": 1,
    "abandoned_carts_30d": 1,
    "email_opens_30d": 5,
    "campaign_clicks_30d": 1,
    "last_visit_days_ago": 15
}

def test_health():
    response = requests.get(f"{BASE_URL}/health")
    assert response.status_code == 200
    assert response.json()["status"] == "ok"
    print("PASS: health check")

def test_predict():
    response = requests.post(f"{BASE_URL}/predict", json=sample_customer)
    assert response.status_code == 200
    data = response.json()
    assert "churn_probability" in data
    assert "predicted_class" in data
    assert "risk_level" in data
    print("PASS: predict endpoint")
    print("Result:", data)

def test_batch_predict():
    response = requests.post(f"{BASE_URL}/batch_predict", json=[sample_customer, sample_customer])
    assert response.status_code == 200
    data = response.json()
    assert data["total"] == 2
    print("PASS: batch predict endpoint")

if __name__ == "__main__":
    test_health()
    test_predict()
    test_batch_predict()
    print("All tests passed!")
'''

with open('/Users/pratikshaupadhyay/Desktop/capstone/part4/tests/test_api.py', 'w') as f:
    f.write(test_py)
print("test_api.py created!")

FileNotFoundError: [Errno 2] No such file or directory: '/Users/pratikshaupadhyay/Desktop/capstone/part4/tests/test_api.py'

In [3]:
# CELL 3 - Create folders and all remaining files
import os

# Create folders
os.makedirs('/Users/pratikshaupadhyay/Desktop/capstone/part4/tests', exist_ok=True)
os.makedirs('/Users/pratikshaupadhyay/Desktop/capstone/part4/app', exist_ok=True)

# Test file
test_py = '''import requests

BASE_URL = "http://localhost:8000"

sample_customer = {
    "recency_days": 45,
    "frequency_180d": 3,
    "monetary_180d": 1500.0,
    "return_rate_180d": 0.1,
    "avg_discount_pct_180d": 0.2,
    "avg_rating_180d": 3.5,
    "category_diversity_180d": 2,
    "ticket_count_90d": 1,
    "negative_ticket_rate_90d": 0.0,
    "avg_resolution_hours_90d": 24.0,
    "days_since_signup": 365,
    "sessions_30d": 3,
    "product_views_30d": 10,
    "cart_adds_30d": 2,
    "wishlist_adds_30d": 1,
    "abandoned_carts_30d": 1,
    "email_opens_30d": 5,
    "campaign_clicks_30d": 1,
    "last_visit_days_ago": 15
}

def test_health():
    response = requests.get(f"{BASE_URL}/health")
    assert response.status_code == 200
    assert response.json()["status"] == "ok"
    print("PASS: health check")

def test_predict():
    response = requests.post(f"{BASE_URL}/predict", json=sample_customer)
    assert response.status_code == 200
    data = response.json()
    assert "churn_probability" in data
    print("PASS: predict endpoint")
    print("Result:", data)

def test_batch_predict():
    response = requests.post(f"{BASE_URL}/batch_predict", json=[sample_customer, sample_customer])
    assert response.status_code == 200
    data = response.json()
    assert data["total"] == 2
    print("PASS: batch predict endpoint")

if __name__ == "__main__":
    test_health()
    test_predict()
    test_batch_predict()
    print("All tests passed!")
'''

with open('/Users/pratikshaupadhyay/Desktop/capstone/part4/tests/test_api.py', 'w') as f:
    f.write(test_py)
print("test_api.py created!")

test_api.py created!


In [4]:
# CELL 4 - Create monitoring plan, responsible use note, requirements and README

import os

# monitoring_plan.md
with open('/Users/pratikshaupadhyay/Desktop/capstone/part4/monitoring_plan.md', 'w') as f:
    f.write("""# Monitoring Plan

## What to Track After Deployment

### 1. Data Drift
- Monitor distribution of recency_days and monetary_180d weekly
- Alert if average recency increases by more than 20 days
- Compare incoming feature distributions to training data monthly

### 2. Prediction Distribution
- Track daily ratio of high/medium/low risk predictions
- Alert if more than 70% of predictions are high risk on any day
- Log all predictions with timestamps to a database

### 3. Business Outcomes
- Compare predicted churners vs actual churners monthly
- Track retention campaign conversion rate by risk segment
- Measure revenue saved from successful interventions

### 4. API Health
- Monitor response time - alert if above 500ms
- Track error rate - alert if above 1%
- Log all 400/500 errors with input payload for debugging

### 5. Retraining Triggers
- Retrain if ROC-AUC drops below 0.80 on recent data
- Retrain every 90 days regardless of performance
- Retrain immediately if churn rate shifts by more than 10%

## Responsible Use Note
- The API output is a risk score, not a final decision
- Never deny service to a customer based on churn score alone
- High risk customers should receive help and retention offers
- Do not use churn score to discriminate by city tier or age group
- Always combine model output with human judgment
- Retention team should review edge cases manually
""")

# requirements.txt
with open('/Users/pratikshaupadhyay/Desktop/capstone/part4/requirements.txt', 'w') as f:
    f.write("""fastapi==0.128.8
uvicorn==0.39.0
pydantic==2.13.4
scikit-learn==1.6.1
pandas==2.3.3
numpy==2.0.2
joblib==1.5.1
requests==2.32.5
pytest==8.4.2
""")

# README.md
with open('/Users/pratikshaupadhyay/Desktop/capstone/part4/README.md', 'w') as f:
    f.write("""# Part 4: FastAPI Churn Scoring Service

## Project Overview
A FastAPI service that loads a trained churn model and returns churn risk predictions for a D2C personal care brand CRM system.

## Repository Structure
- app/main.py - FastAPI application
- tests/test_api.py - API test cases
- model.pkl - Trained Random Forest model
- monitoring_plan.md - Post-deployment monitoring plan
- requirements.txt - Python dependencies

## Setup Instructions

### Install dependencies
pip install -r requirements.txt

### Run the API
cd app
uvicorn main:app --reload

### The API will be available at
http://localhost:8000

## API Endpoints

### GET /health
Returns API health status.
Response: {"status": "ok", "model": "Random Forest Churn Predictor"}

### POST /predict
Accepts one customer feature payload and returns churn risk.
Response includes churn_probability, predicted_class, risk_level, risk_explanation.

### POST /batch_predict
Accepts multiple customers and returns predictions for each.

## Sample Request
POST /predict
{
    "recency_days": 45,
    "frequency_180d": 3,
    "monetary_180d": 1500.0,
    "return_rate_180d": 0.1,
    "avg_discount_pct_180d": 0.2,
    "avg_rating_180d": 3.5,
    "category_diversity_180d": 2,
    "ticket_count_90d": 1,
    "negative_ticket_rate_90d": 0.0,
    "avg_resolution_hours_90d": 24.0,
    "days_since_signup": 365,
    "sessions_30d": 3,
    "product_views_30d": 10,
    "cart_adds_30d": 2,
    "wishlist_adds_30d": 1,
    "abandoned_carts_30d": 1,
    "email_opens_30d": 5,
    "campaign_clicks_30d": 1,
    "last_visit_days_ago": 15
}

## Sample Response
{
    "churn_probability": 0.42,
    "predicted_class": 1,
    "risk_level": "medium",
    "risk_explanation": "Medium churn risk. Recency of 45 days and 3 sessions in last 30 days."
}

## Run Tests
python tests/test_api.py

## Model Notes
- Model: Random Forest Classifier
- Trained on: rfm_modeling_snapshot.csv
- ROC-AUC: 0.878
- Threshold: 0.4
""")

print("All files created!")
print("Files in part4:")
for f in os.listdir('/Users/pratikshaupadhyay/Desktop/capstone/part4'):
    print(" -", f)

All files created!
Files in part4:
 - metrics.json
 - app
 - requirements.txt
 - tests
 - model.pkl
 - README.md
 - monitoring_plan.md
 - .ipynb_checkpoints
 - create_files.ipynb
